Week 3 Assignment – Synthetic Data Generator with Hugging Face Transformers

Author: Silas Wawire
Course: Large Language Models (LLMs)
Week: 3

Project Description

This project implements a Synthetic Data Generation System powered by Hugging Face Transformers.

The system is designed to generate structured synthetic datasets using large language models. It supports:

Text generation using Causal Language Models (e.g., GPT-style models)

Text-to-text generation using Seq2Seq models (e.g., FLAN-T5)

Controlled sampling strategies (temperature, top-k, top-p)

Batch generation for efficiency

Exporting generated outputs to JSON, CSV, and JSONL formats

An interactive Gradio-based web interface for real-time dataset creation

The system can be used for:

Generating Q&A pairs

Creating fine-tuning datasets

Data augmentation

Rapid dataset prototyping

In [8]:
pip install transformers torch gradio pandas

Note: you may need to restart the kernel to use updated packages.


c:\Users\user\llm_engineering\.venv\Scripts\python.exe: No module named pip


In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer
import pandas as pd
import json
import os

class SyntheticDataGenerator:
    def __init__(self, model_name="google/flan-t5-base", model_type="seq2seq"):
        self.model_name = model_name
        self.model_type = model_type
        
        print("Loading model...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        if model_type == "causal":
            self.model = AutoModelForCausalLM.from_pretrained(model_name)
        else:
            self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
        print(f"Model loaded on {self.device}")
    
    def generate(
        self,
        prompt,
        max_length=150,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        num_return_sequences=1
    ):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        
        outputs = self.model.generate(
            **inputs,
            max_length=max_length,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True,
            num_return_sequences=num_return_sequences
        )
        
        results = [
            self.tokenizer.decode(output, skip_special_tokens=True)
            for output in outputs
        ]
        
        return results
    
    def batch_generate(self, prompt, num_samples=5):
        return self.generate(prompt, num_return_sequences=num_samples)

    def export_data(self, data, filename="synthetic_data", format="json"):
        if format == "json":
            with open(f"{filename}.json", "w") as f:
                json.dump(data, f, indent=4)
        
        elif format == "jsonl":
            with open(f"{filename}.jsonl", "w") as f:
                for item in data:
                    f.write(json.dumps(item) + "\n")
        
        elif format == "csv":
            df = pd.DataFrame(data)
            df.to_csv(f"{filename}.csv", index=False)
        
        print(f"Data exported as {filename}.{format}")

Example Usage (Script Mode)

In [10]:
if __name__ == "__main__":
    
    generator = SyntheticDataGenerator(
        model_name="google/flan-t5-base",
        model_type="seq2seq"
    )
    
    prompt = "Generate a question and answer pair about cybersecurity."
    
    results = generator.batch_generate(prompt, num_samples=5)
    
    dataset = []
    for i, text in enumerate(results):
        dataset.append({
            "id": i + 1,
            "content": text
        })
    
    generator.export_data(dataset, filename="qa_dataset", format="json")
    
    print("\nGenerated Samples:\n")
    for r in results:
        print(r)
        print("-" * 50)

Loading model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:  62%|######2   | 619M/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded on cpu
Data exported as qa_dataset.json

Generated Samples:

What are the most common kinds of cybersecurity?
--------------------------------------------------
What is a good way to help protect your network?
--------------------------------------------------
What is the main threat to the Internet?
--------------------------------------------------
What is a common way to connect an iPhone and a laptop?
--------------------------------------------------
What is the purpose of a cyber security plan?
--------------------------------------------------


Gradio Interface (Interactive UI)

In [11]:
import gradio as gr

generator = SyntheticDataGenerator(
    model_name="google/flan-t5-base",
    model_type="seq2seq"
)

def generate_ui(prompt, temperature, samples):
    results = generator.batch_generate(
        prompt,
        num_samples=int(samples)
    )
    return "\n\n".join(results)

interface = gr.Interface(
    fn=generate_ui,
    inputs=[
        gr.Textbox(label="Prompt"),
        gr.Slider(0.1, 1.5, value=0.7, label="Temperature"),
        gr.Slider(1, 10, value=3, step=1, label="Number of Samples")
    ],
    outputs=gr.Textbox(label="Generated Output"),
    title="Synthetic Data Generator"
)

if __name__ == "__main__":
    interface.launch()

Loading model...
Model loaded on cpu
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
